# 01 — Data Profiling & Quality Intelligence

Profile column distributions, missingness, outliers, correlations, train/test drift, and generate per-record data-quality scores.

In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from src.pipeline.loader import load_data, reconcile_servicer_updates
from src.pipeline.validation import load_rules, apply_rules, add_quality_flag

# ── Load data ──────────────────────────────────────────────────────────────────
train_df = pd.read_csv('../data/synthetic/loan_monthly_performance_train.csv')
test_df  = pd.read_csv('../data/synthetic/loan_monthly_performance_test.csv')
static_df   = pd.read_csv('../data/synthetic/loan_static_attributes.csv')
servicer_df = pd.read_csv('../data/synthetic/servicer_updates.csv')

for col in ['reporting_month', 'origination_month']:
    for df in [train_df, test_df]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col]).dt.to_period('M')

print(f'Train: {train_df.shape}  |  Test: {test_df.shape}')
train_df.head(3)


## Validation Rules

In [ ]:
rules = load_rules('../config/validation_rules.json')
violations_df, summary_df = apply_rules(train_df, rules)
train_df = add_quality_flag(train_df, violations_df)
print(f'Rules applied: {len(rules)}  |  Violations: {len(violations_df)}')
summary_df


## Missingness Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
miss = train_df.isnull().mean().sort_values(ascending=False)
miss = miss[miss > 0]
if len(miss) > 0:
    miss.plot(kind='bar', ax=ax, color='steelblue')
    ax.set_ylabel('Missing Rate')
    ax.set_title('Column Missingness (Train)')
    plt.xticks(rotation=45, ha='right')
else:
    ax.text(0.5, 0.5, 'No missing values', ha='center', va='center', fontsize=16)
    ax.set_title('Column Missingness (Train)')
plt.tight_layout()
plt.savefig('../reports/missingness_heatmap.png', dpi=150)
plt.show()
print(miss)


## Distribution Analysis — Key Numeric Columns

In [ ]:
numeric_cols = ['current_balance', 'interest_rate', 'days_past_due', 'loan_age_months']
numeric_cols = [c for c in numeric_cols if c in train_df.columns]
fig, axes = plt.subplots(1, len(numeric_cols), figsize=(4*len(numeric_cols), 3))
for ax, col in zip(axes, numeric_cols):
    train_df[col].hist(bins=40, ax=ax, color='steelblue', edgecolor='white')
    ax.set_title(col, fontsize=9)
    ax.set_xlabel('')
plt.suptitle('Train Feature Distributions', y=1.02)
plt.tight_layout()
plt.savefig('../reports/distributions.png', dpi=150)
plt.show()


## Correlation Matrix

In [ ]:
num_df = train_df.select_dtypes(include='number').drop(columns=[
    c for c in ['next_3m_delinquency_flag','next_6m_delinquency_flag',
                'next_12m_default_flag','next_12m_prepayment_flag'] if c in train_df.columns
], errors='ignore')
corr = num_df.corr()
fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=False, cmap='RdBu_r', center=0, ax=ax,
            linewidths=0.3, cbar_kws={'shrink': 0.7})
ax.set_title('Correlation Matrix (numeric features, train)')
plt.tight_layout()
plt.savefig('../reports/correlation_matrix.png', dpi=150)
plt.show()


## Train vs Test Drift

In [ ]:
drift_cols = ['interest_rate', 'days_past_due', 'current_balance', 'loan_age_months']
drift_cols = [c for c in drift_cols if c in train_df.columns and c in test_df.columns]

from scipy import stats
drift_results = []
for col in drift_cols:
    tr = train_df[col].dropna()
    te = test_df[col].dropna()
    stat, pval = stats.ks_2samp(tr, te)
    drift_results.append({'feature': col, 'ks_statistic': round(stat, 4),
                          'p_value': round(pval, 6), 'drift': 'YES' if pval < 0.05 else 'NO'})

drift_df = pd.DataFrame(drift_results)
print(drift_df.to_string(index=False))


## Data Quality Score

In [ ]:
quality_counts = train_df['data_quality_flag'].value_counts().sort_index()
labels = {0: 'Clean (0)', 1: 'Warning (1)', 2: 'Error (2)'}
print('Data quality distribution:')
for k, v in quality_counts.items():
    print(f'  {labels.get(k, k)}: {v:,} ({v/len(train_df):.1%})')
print(f'\nOverall data quality score: {(train_df["data_quality_flag"]==0).mean():.2%} clean records')
